In [0]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

# On récupére le json source qui est sur le bucket jedha
s3_url = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

#Lecture du fichier JSON
df_steam = spark.read.json(s3_url)

# affichage de la structure des données pour avoir les différents noeuds
df_steam.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:

# à partir du prenmier noeud 'data' on crée des colonnes à partir des enfants
df_flat = df_steam.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.developer").alias("developer"),
    F.col("data.genre").alias("genre"),
    F.col("data.initialprice").alias("initialprice"),
    F.col("data.price").alias("price"),
    F.col("data.discount").alias("discount"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.languages").alias("languages"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.platforms").alias("platforms") 
)

# aperçu des données
display(df_flat.head(10))

id,appid,name,publisher,developer,genre,initialprice,price,discount,release_date,required_age,languages,positive,negative,platforms
10,10,Counter-Strike,Valve,Valve,Action,999,999,0,2000/11/1,0,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",201215,5199,"List(true, true, true)"
1000000,1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999,999,0,2021/05/14,0,"English, Korean, Simplified Chinese",27,5,"List(false, false, true)"
1000010,1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",1999,599,70,2020/10/16,0,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",4032,646,"List(false, false, true)"
1000030,1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999,1999,0,2020/10/14,0,English,1575,115,"List(false, true, true)"
1000040,1000040,细胞战争,DoubleC Games,DoubleC Games,"Action, Casual, Indie, Simulation",199,199,0,2019/03/30,0,Simplified Chinese,0,1,"List(false, false, true)"
1000080,1000080,Zengeon,2P Games,IndieLeague Studio,"Action, Adventure, Indie, RPG",1999,799,60,2019/06/24,0,"Simplified Chinese, English, Traditional Chinese, Japanese, Korean",1018,462,"List(false, true, true)"
1000100,1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,七月九日,"Adventure, Indie, RPG, Strategy",1299,1299,0,2019/01/24,0,"Japanese, Simplified Chinese, Traditional Chinese",18,6,"List(false, false, true)"
1000110,1000110,Jumping Master(跳跳大咖),重庆环游者网络科技,重庆环游者网络科技,"Action, Adventure, Casual, Free to Play, Massively Multiplayer",0,0,0,2019/04/8,0,"English, Simplified Chinese, Traditional Chinese",50,34,"List(false, false, true)"
1000130,1000130,Cube Defender,Simon Codrington,Simon Codrington,"Casual, Indie",299,299,0,2019/01/6,0,English,6,0,"List(false, true, true)"
1000280,1000280,Tower of Origin2-Worm's Nest,Villain Role,Villain Role,"Indie, RPG",1399,1399,0,2021/09/9,0,"English, Simplified Chinese, Traditional Chinese",32,12,"List(false, false, true)"


In [0]:
#  QUESTION 11 : La plupart des jeux sont-ils disponibles sur Windows/Mac/Linux ? 

# Calcul du pourcentage de disponibilité pour chaque plateforme
df_flat.select(
    F.round((F.mean(F.col("platforms.windows").cast("int")) * 100),2).alias("Windows"),
    F.round((F.mean(F.col("platforms.mac").cast("int")) * 100),2).alias("Mac"),
    F.round((F.mean(F.col("platforms.linux").cast("int")) * 100),2).alias("Linux")
).show()

+-------+-----+-----+
|Windows|  Mac|Linux|
+-------+-----+-----+
|  99.97|22.93|15.19|
+-------+-----+-----+



![ELa plupart des jeux sont-ils disponibles sur Windows/Mac/Linux](screen_shot/graphique_question11_part_jeu_plateforme.png)

Réponse : <br>
Sans appel, l'Os Windows est et reste le reflet du marché. <br>
Et dans les faits, non avec à peine 22 et 15 pourcents, Mac et Linux sont délaissés par les éditeurs de jeux.

In [0]:
# QUESTION 12 : Certains genres ont-ils une préférence pour certaines plateformes ?

# ajout d'une colonne par plateformes si elle a la valeur True si non vide
df_platform_genre = df_genre_array.select(
    "genre_list",
    F.when(F.col("platforms.mac") == True, F.lit("mac")).alias("mac"),
    F.when(F.col("platforms.windows") == True, F.lit("windows")).alias("win"),
    F.when(F.col("platforms.linux") == True, F.lit("linux")).alias("lin")
)

#On rassemble ces colonnes dans un tableau 
df_platform_genre = df_platform_genre.withColumn(
    "active_platforms", 
    F.array("win", "mac", "lin")
)

# Explode pour avoir une ligne par genre et par plateforme
df_final = df_platform_genre.select(
    F.explode("active_platforms").alias("platform"),
    F.explode("genre_list").alias("clean_genre")
)

# on ne prend pas les null et vide et on affiche 
df_final.filter(F.col("platform").isNotNull() & (F.col("clean_genre") != "")) \
        .groupBy("platform", "clean_genre") \
        .count() \
        .sort(F.col("count").desc()) \
        .show(20, truncate=False)

+--------+------------+-----+
|platform|clean_genre |count|
+--------+------------+-----+
|windows |Indie       |39676|
|windows |Action      |23755|
|windows |Casual      |22082|
|windows |Adventure   |21427|
|windows |Strategy    |10892|
|windows |Simulation  |10832|
|mac     |Indie       |9935 |
|windows |RPG         |9533 |
|linux   |Indie       |6978 |
|windows |Early Access|6145 |
|mac     |Casual      |5130 |
|mac     |Adventure   |5039 |
|mac     |Action      |4564 |
|windows |Free to Play|3391 |
|linux   |Action      |3379 |
|linux   |Casual      |3305 |
|linux   |Adventure   |3302 |
|mac     |Strategy    |3005 |
|windows |Sports      |2665 |
|mac     |Simulation  |2439 |
+--------+------------+-----+
only showing top 20 rows


Réponse : <br>
Mise à part le fait que Windows soit surreprésenté, et le domaine exclusif des genres Action et RPG. <br>
Coté Mac et Linux ce qui ressort du lot c'est le genre 'Indie'.<br>
